In [0]:
3# ── Silver: fact_appearances — FK validation ─────────────────────
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DateType

bronze_app     = spark.table("football_catalog.bronze.appearances")
silver_games   = spark.table("football_catalog.silver.fact_games")
silver_players = spark.table("football_catalog.silver.dim_players") 

# Cast and clean
app_clean = (bronze_app.select(
    F.col("appearance_id"),
    F.col("game_id").cast(IntegerType()),
    F.col("player_id").cast(IntegerType()),
    F.col("player_club_id").cast(IntegerType()),
    F.to_date(F.col("date"), "yyyy-MM-dd").alias("match_date"),
    F.col("goals").cast(IntegerType()),
    F.col("assists").cast(IntegerType()),
    F.col("yellow_cards").cast(IntegerType()),
    F.col("red_cards").cast(IntegerType()),
    F.col("minutes_played").cast(IntegerType())
).fillna({"goals":0,"assists":0,"yellow_cards":0,"red_cards":0}))

# Get distinct IDs and add a boolean flag
valid_games   = silver_games.select("game_id").distinct().withColumn("_is_valid_game", F.lit(True))
valid_players = silver_players.select("player_id").distinct().withColumn("_is_valid_player", F.lit(True))

# Use LEFT JOINs to flag the data in a single, highly performant pass
flagged_apps = (app_clean
    .join(valid_games, "game_id", "left")
    .join(valid_players, "player_id", "left")
)

# Split the data based on the null flags
good = (flagged_apps
    .filter(F.col("_is_valid_game").isNotNull() & F.col("_is_valid_player").isNotNull())
    .drop("_is_valid_game", "_is_valid_player")
)

bad = (flagged_apps
    .filter(F.col("_is_valid_game").isNull() | F.col("_is_valid_player").isNull())
    .drop("_is_valid_game", "_is_valid_player")
    .withColumn("quarantine_reason", F.lit("FK_VIOLATION"))
)

print(f"Good rows : {good.count():,}")
print(f"Bad rows  : {bad.count():,}")

# Write GOOD data to Silver Fact Table
(good.write.format("delta").mode("overwrite")
 .option("overwriteSchema","true")
 .saveAsTable("football_catalog.silver.fact_appearances"))

# Write BAD data to Quarantine Table
(bad.write.format("delta").mode("append")
 .option("mergeSchema","true")
 .saveAsTable("football_catalog.silver.quarantine"))